# Week 3 — Recurrent Classifiers: BiLSTM and BiGRU

**Goal (syllabus unit III):** the project's first neural network. Every model so far treated an essay as a *bag* of words: TF-IDF counts them, and averaged embeddings blend them into one vector. Both throw away **word order**. A recurrent network reads the essay **one word at a time, in order**, carrying a memory (hidden state) forward, so it can in principle pick up phrasing, sentence rhythm and structure.

Plan:
1. Turn each essay into a sequence of word ids; initialise the embedding layer from the **Week 2 Skip-gram** vectors.
2. Build a **bidirectional LSTM** classifier, sanity-check that it can memorise one batch, then train it.
3. Train two comparisons: a **BiGRU** (a simpler recurrent cell) and a **BiLSTM with random embeddings** (to see whether the Week 2 vectors help).
4. Score all of them with `evaluate` **and** `evaluate_on_topic` (see `06_topic_check`), next to TF-IDF.

Hardware note: this runs on **CPU** (the laptop's RTX 3050 has no working driver yet), so the models are kept small.

In [1]:
import sys
sys.path.append('..')

import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gensim.models import KeyedVectors
from gensim.utils import simple_preprocess
from sklearn.metrics import roc_auc_score

from src.data import load_splits
from src.evaluate import evaluate, evaluate_on_topic, COLUMNS

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(8)
pd.set_option('display.width', 200)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

torch 2.12.0+cu130 | CUDA available: False


## Step 1 — From text to sequences of ids

A neural network needs numbers, so:
1. **Tokenize** exactly as in `05` (`simple_preprocess`: lowercase, punctuation removed).
2. **Vocabulary** = the 16,424 words the Skip-gram model knows, plus two special tokens:
   - `<pad>` (id 0): filler so essays of different lengths can be stacked into one batch tensor;
   - `<unk>` (id 1): any word not in the vocabulary (rare words, typos seen < 5 times in training).
3. **Truncate** to the first `MAX_LEN = 384` tokens. Recurrent nets are slow on CPU, because each step waits for the previous one. 384 covers the median essay in full; for long essays the model sees the opening ~384 words, which carry plenty of style. (A side effect: it can't use "this essay is very long" beyond 384 tokens, which removes part of the length shortcut.)

In [2]:
MAX_LEN = 384
data = load_splits()
data['tokens'] = data['text'].map(lambda t: simple_preprocess(t, min_len=1, max_len=30))

kv = KeyedVectors.load('../models/w2v_skipgram.kv')
itos = ['<pad>', '<unk>'] + list(kv.index_to_key)          # id -> word
stoi = {w: i for i, w in enumerate(itos)}                   # word -> id
data['ids'] = data['tokens'].map(lambda toks: [stoi.get(w, 1) for w in toks[:MAX_LEN]] or [1])

n_tok = data['tokens'].map(len)
print(f"vocabulary: {len(itos):,} ids")
print(f"essays fully inside {MAX_LEN} tokens: {(n_tok <= MAX_LEN).mean() * 100:.0f}%")
print(f"<unk> rate on val: {np.mean([i == 1 for ids in data.loc[data['split'] == 'val', 'ids'] for i in ids]) * 100:.2f}%")
print("\nexample:", data['tokens'].iloc[0][:8], "->", data['ids'].iloc[0][:8])

train, val, test = (data[data['split'] == s] for s in ['train', 'val', 'test'])
ho_prompt, ho_gen = data[data['split'] == 'heldout_prompt'], data[data['split'] == 'heldout_generator']

vocabulary: 16,426 ids
essays fully inside 384 tokens: 58%
<unk> rate on val: 0.67%

example: ['phones', 'modern', 'humans', 'today', 'are', 'always', 'on', 'their'] -> [113, 1371, 343, 380, 16, 169, 19, 24]


**Embedding matrix.** Row *i* is the starting vector for word id *i*: the Skip-gram vector for real words, zeros for `<pad>`, and a small random vector for `<unk>`. The network will **fine-tune** these during training, adjusting them for this task.

For the random-embedding comparison, build a matrix of the same shape filled with random numbers of the same scale.

In [3]:
dim = kv.vector_size
emb_skipgram = np.zeros((len(itos), dim), dtype=np.float32)
emb_skipgram[2:] = kv.vectors
scale = kv.vectors.std()
rng = np.random.default_rng(SEED)
emb_skipgram[1] = rng.normal(0, scale, dim)

emb_random = rng.normal(0, scale, (len(itos), dim)).astype(np.float32)
emb_random[0] = 0
print(emb_skipgram.shape, f"(vector std {scale:.3f})")

(16426, 100) (vector std 0.245)


## Step 2 — Batching

Training runs on **mini-batches** of 64 essays. Essays in a batch are padded with `<pad>` to the longest one, so essays of very different lengths would waste most of the batch on padding. **Bucketing:** sort essays by length, cut the sorted list into batches (so each batch holds similar lengths), then shuffle the *order of the batches* every epoch, so training still sees them in a random order.

In [4]:
def batches(df, batch_size=64, shuffle=False, seed=0):
    ids, labels = df['ids'].tolist(), df['label'].to_numpy()
    order = np.argsort([len(x) for x in ids], kind='stable')
    chunks = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    if shuffle:
        np.random.default_rng(seed).shuffle(chunks)
    for chunk in chunks:
        lengths = torch.tensor([len(ids[i]) for i in chunk])
        x = torch.zeros(len(chunk), int(lengths.max()), dtype=torch.long)    # 0 = <pad>
        for row, i in enumerate(chunk):
            x[row, :len(ids[i])] = torch.tensor(ids[i])
        yield x, lengths, torch.tensor(labels[chunk], dtype=torch.float32), chunk

x, lengths, y, _ = next(batches(train.sample(256, random_state=SEED), shuffle=True))
print("one batch:", tuple(x.shape), "| lengths", lengths[:5].tolist(), "| labels", y[:5].tolist())

real = sum(len(i) for i in train['ids'])
padded = sum(int(l.max()) * len(l) for _, l, _, _ in batches(train, 64))
print(f"padding with bucketing: {(padded - real) / padded * 100:.2f}% of all positions in the training batches")

one batch: (64, 384) | lengths [362, 362, 364, 365, 365] | labels [0.0, 1.0, 0.0, 0.0, 0.0]


padding with bucketing: 0.10% of all positions in the training batches


## Step 3 — The model

```
word ids ─► Embedding (100-d, Skip-gram init) ─► dropout
        ─► bidirectional LSTM (64 units each way) ─► max-pool over time ─► dropout ─► Linear ─► 1 logit
```

- **LSTM** (Long Short-Term Memory): a recurrent cell with a *cell state* and three **gates** (input, forget, output). The gates learn what to write, keep and expose at each step, which lets information survive over long spans and avoids the vanishing gradients of a plain RNN.
- **GRU** (Gated Recurrent Unit): a lighter cell with two gates (update, reset) and no separate cell state. Fewer parameters, often about as good, and usually faster on GPU. (On this CPU it turned out ~2.6× *slower*; see Step 5.)
- **Bidirectional:** one LSTM reads left→right, another right→left, and their outputs are concatenated (128 numbers per word), so every position sees context from both sides.
- **Max-pooling over time:** for each of the 128 features, take its largest value anywhere in the essay. This asks "did a tell-tale pattern occur *somewhere*?", which suits detection better than relying only on the final hidden state after 384 steps. Padded positions are set to −∞ (a *mask*) so they can never be the max.
- **Padding, an engineering trade-off.** The textbook way to stop an RNN from reading `<pad>` is *packing* (`pack_padded_sequence`). Measured on this CPU, packing made each batch **12× slower** (8.7 s vs 0.7 s), because PyTorch's fast fused LSTM kernel can't run on packed input. Training would take hours. Instead, bucketing (Step 2) keeps padding to a tiny fraction of positions, and the max-pool mask hides it. The only remaining effect: the right→left LSTM reads a few zero-vector `<pad>` steps before the real text, which is negligible at that padding rate.
- **Output:** one number (a *logit*). Sigmoid(logit) = probability of AI. The loss is **binary cross-entropy**.
- **Dropout (0.3):** randomly zeroes 30% of values during training, so the network can't depend on any single feature (regularisation, like `C` in `04`).

In [5]:
class RNNClassifier(nn.Module):
    def __init__(self, emb_matrix, cell='lstm', hidden=64, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.tensor(emb_matrix), freeze=False, padding_idx=0)
        rnn = nn.LSTM if cell == 'lstm' else nn.GRU
        self.rnn = rnn(emb_matrix.shape[1], hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * hidden, 1)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))                                        # (batch, time, 100)
        h, _ = self.rnn(e)                                                # (batch, time, 128)
        is_pad = (x == 0).unsqueeze(-1)                                   # (batch, time, 1)
        pooled = h.masked_fill(is_pad, float('-inf')).max(dim=1).values   # (batch, 128)
        return self.out(self.drop(pooled)).squeeze(1)                     # (batch,) logits

m = RNNClassifier(emb_skipgram)
n_params = sum(p.numel() for p in m.parameters())
n_emb = m.emb.weight.numel()
print(f"parameters: {n_params:,} total = {n_emb:,} embedding + {n_params - n_emb:,} LSTM/output")
print("output on the example batch:", m(x, lengths)[:4].detach().numpy().round(3))

parameters: 1,727,721 total = 1,642,600 embedding + 85,121 LSTM/output


output on the example batch: [-0.055 -0.178 -0.023  0.004]


Most parameters are the embedding table (one 100-d vector per word); the LSTM itself is small.

## Step 4 — Sanity check: can it memorise one batch?

The standard first test for a new network: train repeatedly on a **single batch of 64 essays**. A correctly wired model with enough capacity should drive the loss close to 0. If it can't even memorise 64 examples, there's a bug (wrong labels, broken gradients, a shape mix-up), and full training would be wasted time.

In [6]:
torch.manual_seed(SEED)
m = RNNClassifier(emb_skipgram, dropout=0.0)
opt = torch.optim.Adam(m.parameters(), lr=3e-3)
x, lengths, y, _ = next(batches(train.sample(64, random_state=SEED)))
for step in range(61):
    loss = nn.functional.binary_cross_entropy_with_logits(m(x, lengths), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 15 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")
assert loss.item() < 0.05, "could not overfit one batch: check the model"
print("Sanity check passed: the model can memorise a batch.")

step   0  loss 0.6881


step  15  loss 0.3229


step  30  loss 0.0218


step  45  loss 0.1738


step  60  loss 0.0149
Sanity check passed: the model can memorise a batch.


## Step 5 — Training

- **Optimizer: Adam** (learning rate 1e-3): gradient descent with a per-parameter adaptive step size, the usual default for RNNs.
- **Gradient clipping** (max norm 1.0): RNN gradients can occasionally explode, because they're multiplied through many time steps. Clipping caps the size of each update.
- **Epochs:** 3 passes over the training set. After each one, measure **val AUC** and keep the weights from the best epoch (*early stopping by checkpoint*). Test and the held-out slices are not touched.

`predict` runs the model without gradients (`torch.no_grad()`, dropout off via `model.eval()`) and returns probabilities in the dataframe's row order.

In [7]:
def predict(model, df, batch_size=256):
    model.eval()
    probs = np.empty(len(df), dtype=np.float32)
    with torch.no_grad():
        for x, lengths, _, chunk in batches(df, batch_size):
            probs[chunk] = torch.sigmoid(model(x, lengths)).numpy()
    return probs

def train_model(emb_matrix, cell, epochs=3, lr=1e-3, seed=SEED):
    torch.manual_seed(seed)
    model = RNNClassifier(emb_matrix, cell=cell)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_auc, best_state, log = -1, None, []
    for epoch in range(1, epochs + 1):
        t0, model_losses = time.time(), []
        model.train()
        for x, lengths, y, _ in batches(train, 64, shuffle=True, seed=seed + epoch):
            loss = nn.functional.binary_cross_entropy_with_logits(model(x, lengths), y)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            model_losses.append(loss.item())
        val_auc = roc_auc_score(val['label'], predict(model, val))
        log.append({'epoch': epoch, 'train_loss': np.mean(model_losses), 'val_auc': val_auc,
                    'minutes': (time.time() - t0) / 60})
        print(f"  epoch {epoch}: train loss {log[-1]['train_loss']:.4f} | val AUC {val_auc:.4f} | {log[-1]['minutes']:.1f} min")
        if val_auc > best_auc:
            best_auc, best_state = val_auc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, pd.DataFrame(log)

Train the three variants. This is the slow cell (~10 min per model on CPU).

In [8]:
variants = {
    'BiLSTM (Skip-gram init)': (emb_skipgram, 'lstm'),
    'BiGRU (Skip-gram init)':  (emb_skipgram, 'gru'),
    'BiLSTM (random init)':    (emb_random, 'lstm'),
}
trained, logs = {}, {}
for name, (emb, cell) in variants.items():
    print(name)
    trained[name], logs[name] = train_model(emb, cell)
    torch.save(trained[name].state_dict(), f"../models/{name.split(' ')[0].lower()}_{'sg' if emb is emb_skipgram else 'rand'}.pt")
pd.concat(logs, names=['model']).round(4)

BiLSTM (Skip-gram init)


  epoch 1: train loss 0.3146 | val AUC 0.9949 | 4.2 min


  epoch 2: train loss 0.0745 | val AUC 0.9967 | 4.2 min


  epoch 3: train loss 0.0389 | val AUC 0.9988 | 4.1 min
BiGRU (Skip-gram init)


  epoch 1: train loss 0.2849 | val AUC 0.9966 | 12.1 min


...

  epoch 3: train loss 0.0202 | val AUC 0.9995 | 10.4 min
BiLSTM (random init)


  epoch 1: train loss 0.3370 | val AUC 0.9958 | 3.8 min


  epoch 2: train loss 0.0676 | val AUC 0.9981 | 4.0 min


  epoch 3: train loss 0.0346 | val AUC 0.9990 | 4.0 min


epoch  train_loss  val_auc  minutes
model                                                         
BiLSTM (Skip-gram init) 0      1      0.3146   0.9949   4.2416
                        1      2      0.0745   0.9967   4.2196
                        2      3      0.0389   0.9988   4.1043
BiGRU (Skip-gram init)  0      1      0.2849   0.9966  12.0947
                        1      2      0.0417   0.9987  10.9129
                        2      3      0.0202   0.9995  10.3945
BiLSTM (random init)    0      1      0.3370   0.9958   3.7907
                        1      2      0.0676   0.9981   3.9805
                        2      3      0.0346   0.9990   4.0116

Things to read in the log:
- **Skip-gram init vs random init:** the expectation was a head start from pretrained embeddings. **There wasn't one:** random init reached val AUC 0.9958 after epoch 1 vs 0.9949 for Skip-gram, and both ended at ~0.999. With 28k essays (≈10M tokens) the network learns useful embeddings from scratch within one epoch, and our Skip-gram vectors were trained on the same essays, so they add no new knowledge. (Pretrained vectors matter most when labelled data is small or the vectors come from a much larger corpus.)
- **LSTM vs GRU:** quality is very close, but the GRU took **10–12 min per epoch vs ~4 min** for the LSTM. On CPU, PyTorch runs the LSTM through a heavily optimised fused routine that its GRU doesn't get. "GRU is faster" is a statement about operation counts and GPUs, not a guarantee.
- **Val AUC was still rising at epoch 3** for every model, so they're probably slightly under-trained. More epochs is a cheap improvement once a GPU is available.

## Step 6 — Score on the final sets

Run each model once over every essay, then score with the shared functions: all essays, then on-topic only.

In [9]:
probs = {}
for name, model in trained.items():
    t0 = time.time()
    probs[name] = predict(model, data)
    print(f"{name}: scored {len(data):,} essays in {time.time() - t0:.0f}s")

rows_of = lambda d: data.index.get_indexer(d.index)
sets = (test, ho_prompt, ho_gen)
all_rows = pd.DataFrame([evaluate(lambda d, p=p: p[rows_of(d)], 0.5, n, *sets) for n, p in probs.items()])
fair_rows = pd.DataFrame([evaluate_on_topic(lambda d, p=p: p[rows_of(d)], 0.5, n, *sets) for n, p in probs.items()])

BiLSTM (Skip-gram init): scored 44,864 essays in 78s


BiGRU (Skip-gram init): scored 44,864 essays in 147s


BiLSTM (random init): scored 44,864 essays in 90s


Put the RNNs next to the earlier models. TF-IDF and averaged-Skip-gram rows come from `results/04`–`06` (all-essay rows from `04`/`05`, on-topic rows from `06`).

In [10]:
key = ['test_f1', 'test_auc', 'heldout_prompt_f1', 'heldout_prompt_auc', 'heldout_gen_detection', 'heldout_gen_auc']
prev_all = pd.concat([pd.read_csv('../results/04_baseline.csv', index_col=0),
                      pd.read_csv('../results/05_embeddings.csv', index_col=0)])
prev_fair = pd.read_csv('../results/06_on_topic.csv', index_col=0)
keep_prev = ['TF-IDF + LinearSVM (C=1)', 'avg Skip-gram + LogReg (C=10)', 'avg Skip-gram + LogReg']
prev_all = prev_all[prev_all.index.isin(keep_prev)].rename(index=lambda s: s.replace(' (C=10)', ''))
prev_fair = prev_fair[prev_fair.index.isin(keep_prev)]

compare = pd.concat({'all essays': pd.concat([prev_all, all_rows])[key],
                     'on-topic only': pd.concat([prev_fair, fair_rows])[key]}, axis=1)
compare.round(3)

all essays                                                                                     on-topic only                                                \
                            test_f1 test_auc heldout_prompt_f1 heldout_prompt_auc heldout_gen_detection heldout_gen_auc       test_f1 test_auc heldout_prompt_f1 heldout_prompt_auc   
TF-IDF + LinearSVM (C=1)      0.995    1.000             0.973              0.997                 0.909           1.000         0.994    1.000             0.965              0.997   
avg Skip-gram + LogReg        0.967    0.995             0.839              0.949                 0.838           0.980         0.942    0.995             0.787              0.935   
BiLSTM (Skip-gram init)       0.983    0.999             0.867              0.965                 0.925           0.994         0.973    0.999             0.822              0.953   
BiGRU (Skip-gram init)        0.989    1.000             0.831              0.950                 0.964           0.998         0.974    1.000             0.775              0.934   
BiLSTM (random init)          0.988    0.999             0.876              0.966                 0.900           0.995         0.973    0.998             0.835              0.957   

                                                                
                         heldout_gen_detection heldout_gen_auc  
TF-IDF + LinearSVM (C=1)                 0.880           1.000  
avg Skip-gram + LogReg                   0.804           0.976  
BiLSTM (Skip-gram init)                  0.908           0.992  
BiGRU (Skip-gram init)                   0.955           0.998  
BiLSTM (random init)                     0.870           0.994

## Step 7 — Where does the best RNN fail?

Error rates by source on `test` (false-positive rate for human sources, miss rate for AI sources) and on the held-out generators, then a few real mistakes.

In [11]:
best = max(trained, key=lambda n: fair_rows.loc[n, 'heldout_prompt_auc'])
p = probs[best]
print(f"Best on on-topic held-out prompts: {best}\n")
for name, d in [('TEST', test), ('HELD-OUT GENERATORS', ho_gen)]:
    out = d.assign(p_ai=p[rows_of(d)], wrong=lambda t: (t['p_ai'] >= 0.5) != t['label'])
    print(name, "- % wrong by source")
    print(out.groupby(['label', 'source'])['wrong'].mean().mul(100).round(1).to_string(), "\n")

t = test.assign(p_ai=p[rows_of(test)])
for title, rows in [("HUMAN essays scored most AI-like", t[t['label'] == 0].nlargest(2, 'p_ai')),
                    ("AI essays scored most human-like", t[t['label'] == 1].nsmallest(2, 'p_ai'))]:
    print(f"===== {title} =====")
    for _, r in rows.iterrows():
        print(f"[p_ai={r['p_ai']:.2f} | {r['source']} | {r['prompt_name']} | on_topic={r['on_topic']}]")
        print(r['text'][:350].replace('\n', ' '), '...\n')

Best on on-topic held-out prompts: BiLSTM (random init)

TEST - % wrong by source
label  source                            
0      persuade_corpus                       0.7
       train_essays                          1.6
1      NousResearch/Llama-2-7b-chat-hf       0.0
       chat_gpt_moth                         0.0
       cohere-command                        3.4
       kingki19_palm                         0.0
       llama2_chat                           0.5
       llama_70b_v1                          2.0
       mistral7binstruct_v1                  0.0
       mistral7binstruct_v2                  3.9
       mistralai/Mistral-7B-Instruct-v0.1    0.0
       palm-text-bison1                      0.0
       radek_500                             0.0
       radekgpt4                             0.0 

HELD-OUT GENERATORS - % wrong by source
label  source           
1      darragh_claude_v6    12.6
       darragh_claude_v7    13.7
       falcon_180b_v1        4.3 

===== HUMAN essays sco

## Step 8 — Save

Results go to `results/07_bilstm.csv`, with a `subset` column (`all` / `on_topic`) so both views live in one file. Weights were saved to the git-ignored `models/` folder in Step 5.

In [12]:
out = pd.concat([all_rows.assign(subset='all'), fair_rows.assign(subset='on_topic')])
out[['subset'] + COLUMNS].round(4).to_csv('../results/07_bilstm.csv')
print(open('../results/07_bilstm.csv').read())

,subset,test_acc,test_precision,test_recall,test_f1,test_auc,heldout_prompt_f1,heldout_prompt_auc,heldout_gen_detection,heldout_gen_auc
BiLSTM (Skip-gram init),all,0.9886,0.9835,0.9835,0.9835,0.9991,0.8674,0.9645,0.9254,0.9935
BiGRU (Skip-gram init),all,0.9926,0.9845,0.9942,0.9893,0.9998,0.8314,0.9505,0.9644,0.9983
BiLSTM (random init),all,0.9914,0.986,0.9893,0.9876,0.9988,0.8758,0.9664,0.8997,0.9955
BiLSTM (Skip-gram init),on_topic,0.9905,0.9589,0.9873,0.9729,0.9987,0.8215,0.9533,0.9082,0.992
BiGRU (Skip-gram init),on_topic,0.9909,0.9609,0.9873,0.9739,0.9996,0.7749,0.9337,0.9546,0.9979
BiLSTM (random init),on_topic,0.9905,0.9647,0.981,0.9727,0.9976,0.8353,0.9567,0.8701,0.9943



## Key takeaways for Week 4

- **First neural networks in the project:** BiLSTM and BiGRU (~85k recurrent/output parameters + a 1.6M-parameter embedding table), trained on CPU in ~12 min (LSTM) to ~33 min (GRU). Padding cost only 0.10% of positions thanks to bucketing, so masking instead of packing (12× faster on this CPU) was safe.
- **Recurrent models beat averaged embeddings clearly** (on-topic held-out prompts AUC 0.93–0.96 vs 0.935 for averaged Skip-gram and 0.48 for averaged GloVe), so word order and local phrasing matter.
- **But TF-IDF still wins on unseen topics:** on-topic held-out-prompt AUC 0.997 vs 0.957 for the best RNN (random-init BiLSTM), and F1 0.965 vs 0.835. With ~28k essays and 3 epochs, a small RNN overfits more to topic-specific wording than a linear model on 200k sparse n-grams.
- **RNNs win on unseen generators:** on-topic detection at the default threshold is **95.5% (BiGRU)** and 90.8% (BiLSTM) vs 88.0% for TF-IDF. Claude essays are still the hardest (12–14% missed by the best-on-prompts BiLSTM).
- **Pretrained Skip-gram initialisation didn't help** (no head start, same final quality), because the vectors came from the same training essays.
- **Recurring failure cases, shared with TF-IDF:** (1) polished, formal *human* essays ("Nowadays, technology and science became one of important parts in human life…") get flagged as AI, which is the fairness risk for the report; (2) a few `mistral7binstruct_v2` "AI" essays read exactly like student writing, with grammar slips ("there is some classes"). Every model so far misses them, which suggests residual label noise in that source rather than a model weakness.
- **For Week 4 (BERT/RoBERTa):** fine-tuning a transformer on CPU is impractical, so fix the NVIDIA driver (RTX 3050) or use a free Kaggle/Colab GPU. Compare on the same rows, especially on-topic held-out prompts (TF-IDF 0.997) and held-out generators (BiGRU 95.5%).
